In [ ]:
pip install pandas numpy google-cloud-bigquery pandas-gbq prophet

In [ ]:
import pandas as pd
import pandas_gbq

# 1. The SQL Query
query = """
SELECT 
    product_sk,
    CAST(order_timestamp AS DATE) AS sale_date,
    SUM(fulfilled_qty) AS daily_units_sold
FROM `mart.fact_sales`
GROUP BY 
    product_sk, 
    sale_date
ORDER BY 
    sale_date ASC
"""

# project ID beolow (removed it so no one sees it)
project_id = 'your_google_project_id_here'

print("Knocking on BigQuery's door... Downloading data...")

df = pandas_gbq.read_gbq(query, project_id=project_id)

print("Download complete!")

df.head()

In [ ]:
from prophet import Prophet
import warnings
import logging
import pandas as pd

# ==========================================================
# 1. I SHUT THE AI UP
# The AI talks too much while it works and prints out 
# thousands of useless warning messages. I turn this off 
# so it doesn't freeze my computer screen.
# ==========================================================
warnings.filterwarnings('ignore')
logging.getLogger('prophet').setLevel(logging.WARNING)

# ==========================================================
# 2. I RENAME COLUMNS FOR THE PICKY AI
# The AI refuses to read columns named 'sale_date' or 
# 'daily_units_sold'. I strictly rename the date to 'ds' 
# and the target numbers to 'y'. I also ensure Pandas 
# explicitly knows 'ds' is a date object.
# ==========================================================
prophet_df = df.rename(columns={'sale_date': 'ds', 'daily_units_sold': 'y'})
prophet_df['ds'] = pd.to_datetime(prophet_df['ds'])

# ==========================================================
# 3. I CREATE AN EMPTY BUCKET
# I am going to predict one product at a time. 
# I need an empty bucket to throw my answers into as I 
# finish each one.
# ==========================================================
all_predictions = []

# ==========================================================
# 4. I GET MY GROCERY LIST OF PRODUCTS
# The AI gets confused if I hand it every product at the 
# same time. I extract a list of unique products so I can 
# feed them to the AI one by one.
# ==========================================================
unique_products = prophet_df['product_sk'].unique()
print(f"Starting AI forecast for {len(unique_products)} products. This might take a minute...")

for sku in unique_products:
    
    # I grab only the past sales history for THIS specific product.
    sku_history = prophet_df[prophet_df['product_sk'] == sku].copy()
    
    # ==========================================================
    # 5. I FILL IN THE MISSING ZEROES (SENIOR FIX)
    # SQL only logs days where sales happened. I use Pandas 
    # to stretch my history into a continuous daily calendar 
    # and fill the empty days with 0 sales so the AI doesn't 
    # over-predict my future demand.
    # ==========================================================
    sku_history = sku_history.set_index('ds').resample('D').sum().fillna(0).reset_index()
    sku_history['product_sk'] = sku # I put the product name back after the math
    
    # ==========================================================
    # 6. I SKIP NEW PRODUCTS
    # If a product has only been sold for 1 day, the AI cannot 
    # find a pattern. It needs at least 2 days of history, 
    # otherwise the math crashes. 
    # ==========================================================
    if len(sku_history) < 2:
        continue
        
    # ==========================================================
    # 7. I TURN ON THE AI BRAIN
    # I turn off 'yearly_seasonality' (like Christmas spikes) 
    # because I don't have multiple years of data yet. If I 
    # leave it on, the AI gets confused.
    # ==========================================================
    model = Prophet(daily_seasonality=False, yearly_seasonality=False) 
    
    # ==========================================================
    # 8. I TEACH THE AI
    # I hand the AI my past sales data for this product so 
    # it can study the pattern.
    # ==========================================================
    model.fit(sku_history)
    
    # ==========================================================
    # 9. I GET A BLANK CALENDAR (AND PLUG THE HISTORY LEAK)
    # I generate a blank calendar for the next 30 days. 
    # By adding `include_history=False`, I force the AI to ONLY 
    # generate future days so I don't accidentally upload the 
    # past into my future forecast table.
    # ==========================================================
    future_calendar = model.make_future_dataframe(periods=30, include_history=False)
    
    # ==========================================================
    # 10. I MAKE THE GUESS
    # I force the AI to look at the blank calendar and fill 
    # in the future sales.
    # ==========================================================
    forecast = model.predict(future_calendar)
    
    # ==========================================================
    # 11. I THROW AWAY THE JUNK MATH
    # The AI spits out 20 columns of crazy statistical math I 
    # don't care about. I only keep the Date ('ds') and the 
    # AI's predicted guess ('yhat').
    # ==========================================================
    forecast = forecast[['ds', 'yhat']].copy()
    
    # I put the product name back on the paper so I know what product this is.
    forecast['product_sk'] = sku  
    
    # I throw this finished prediction into my empty bucket.
    all_predictions.append(forecast)

# ==========================================================
# 12. I DUMP THE BUCKET ONTO THE TABLE
# I take all the hundreds of separate product predictions inside 
# my bucket and glue them together into one giant spreadsheet.
# ==========================================================
final_forecast_df = pd.concat(all_predictions)

# ==========================================================
# 13. I RENAME COLUMNS BACK TO NORMAL ENGLISH
# I change the weird AI names ('ds' and 'yhat') back to normal 
# words for my BigQuery database.
# ==========================================================
final_forecast_df = final_forecast_df.rename(columns={'ds': 'forecast_date', 'yhat': 'predicted_sales_qty'})

# ==========================================================
# 14. I FIX IMPOSSIBLE AI MATH (REAL WORLD CHECK)
# The AI is just a math calculator. If a product is dying, 
# it might guess I will sell "-5" boxes. I cannot sell 
# negative boxes. `.clip(lower=0)` forces any negative guess 
# to be a zero.
#
# I also cannot sell "1.5" physical boxes. I force it to 
# round up or down to a whole box using `.round()`.
# ==========================================================
final_forecast_df['predicted_sales_qty'] = final_forecast_df['predicted_sales_qty'].clip(lower=0)
final_forecast_df['predicted_sales_qty'] = final_forecast_df['predicted_sales_qty'].round()

print("AI Forecasting Complete! 🔮")
final_forecast_df.head(10)

In [ ]:
# ==========================================================
# STATION 3: THE SHIPPING ROOM
# The AI has finished. Now we need to push this brand new
# "future" spreadsheet back up into our Google BigQuery 
# database so our BI dashboard can read it.
# ==========================================================
print("Uploading AI predictions to BigQuery...")

# The name of the new table we are creating in BigQuery
destination_table = 'mart.ai_sales_forecast'

# Push the dataframe up to the cloud!
import pandas_gbq

pandas_gbq.to_gbq(
    dataframe=final_forecast_df,
    destination_table=destination_table,
    project_id=project_id,
    if_exists='replace'
    # If we run this script again tomorrow, overwrite the old predictions
)

print("Upload complete! The future is now in your database. 🚀")

# 🔮 AI Demand Forecasting Engine

**Author:** [AFTAB KHAN]

**Purpose:** Pulls historical daily sales from BigQuery `mart.fact_sales`, feeds the data into Meta's Prophet Machine Learning algorithm, and generates a 30-day future sales prediction. The AI forecast is then pushed back into BigQuery as `mart.ai_sales_forecast`.